# LaTeX export — figures and tables for the thesis document

Exports a figure and a table straight into the sibling thesis repository, so a chart in the written thesis is a regenerated artifact rather than a screenshot.

**Inputs:** joined train/test feature artifacts and their metadata contract, plus the corresponding all-station observation artifacts (Stages 2–3 must have run)
**Outputs:** `../uas-master-thesis/figures/*.pdf` and `../uas-master-thesis/tables/*.tex` — **written outside this repository**

This notebook is not part of the `01` → `06` chain and has no `make` target; run it by hand while writing. The thesis preamble needs `\usepackage{booktabs}` and `\usepackage{tabularx}` for the exported table to compile.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Pins the same Stage-3 artifact paths and cohort constants every stage-4 notebook uses. The completeness figure reads the joined feature artifacts, so it includes only stations retained by the target-range overlap filter. `THESIS_TEXTWIDTH_IN` is the thesis body text width (456.25555 pt): authoring the figure at that width means `width=\textwidth` scales it by 1.0 and the in-figure fonts land at their intended size.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pandas as pd
from IPython.display import display

from src.config import (
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MATPLOTLIB_STYLE,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    THESIS_TEXTWIDTH_IN,
    WEATHER_VARIABLES,
)
plt.style.use(MATPLOTLIB_STYLE)
from src.dataset import load_joined_dataset
from src.latex_export import save_figure, save_table

PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
raw_train_path = PROCESSED_DIR / "all_stations_train.parquet"
raw_test_path = PROCESSED_DIR / "all_stations_test.parquet"
WATER_LEVEL_COLUMN = f"{TARGET_STATION_ID}__water_level"

## Load the joined dataset

`load_joined_dataset()` is the repo's only sanctioned entry point into the Stage-3 artifacts: it validates the horizon, target station, and column contracts before returning anything, so a drifted contract fails here instead of silently producing a figure of the wrong thing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
water_level = dataset.target_context_series[WATER_LEVEL_COLUMN]
water_level_columns = [
    column
    for column in dataset.contract.predictor_columns
    if column.endswith("__water_level")
]

## Figure — full-history water level and alarm threshold

The full raw train and sealed-test observation artifacts are combined before aggregation, so this figure covers the entire available period rather than only the eligible modeling cohort. The daily mean water level and weekly precipitation total share one time axis; separate y-axes preserve their different units.

In [ ]:
PRECIPITATION_COLUMN = f"{TARGET_STATION_ID}__precipitation"
full_history = pd.concat(
    [
        pd.read_parquet(
            raw_train_path,
            columns=["timestamp", WATER_LEVEL_COLUMN, PRECIPITATION_COLUMN],
        ),
        pd.read_parquet(
            raw_test_path,
            columns=["timestamp", WATER_LEVEL_COLUMN, PRECIPITATION_COLUMN],
        ),
    ],
    ignore_index=True,
).assign(timestamp=lambda frame: pd.to_datetime(frame["timestamp"], utc=True))
full_history = full_history.sort_values("timestamp").set_index("timestamp")
split_timestamp = pd.to_datetime(
    pd.read_parquet(raw_test_path, columns=["timestamp"])["timestamp"],
    utc=True,
).min()
daily_water_level = full_history[WATER_LEVEL_COLUMN].resample("D").max()
weekly_precipitation = full_history[PRECIPITATION_COLUMN].resample("W").sum()

fig, water_axis = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55)
)
precipitation_axis = water_axis.twinx()
daily_line, = water_axis.plot(
    daily_water_level.index, daily_water_level, label="Daily max water level"
)
threshold_line = water_axis.axhline(
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    color="C2",
    linestyle="--",
    label=f"Alarm threshold ({WATER_LEVEL_ALARM_THRESHOLD_CM:.0f} cm)",
)
weekly_bars = precipitation_axis.bar(
    weekly_precipitation.index,
    weekly_precipitation,
    width=10,
    color="C1",
    alpha=0.8,
    label="Total weekly precipitation",
)
split_line = water_axis.axvline(
    split_timestamp,
    color="black",
    linestyle=":",
    label="Train/test split",
)
water_axis.set_xlabel("Date")
water_axis.set_ylabel("Water level (cm)")
precipitation_axis.set_ylabel("Weekly precip. (mm)")
water_axis.set_title(f"Korneuburg water level ({TARGET_STATION_ID})")
water_axis.grid(alpha=0.25)
fig.tight_layout(rect=(0, 0.18, 1, 1))
fig.legend(
    handles=[daily_line, threshold_line, weekly_bars, split_line],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2
)
z_score = (weekly_precipitation.max() - weekly_precipitation.mean()) / weekly_precipitation.std()

print(f"Highest weekly precipitation: {weekly_precipitation.max()} mm (appeared {weekly_precipitation.idxmax()})")
print(f"Mean weekly precipitation: {weekly_precipitation.mean():.2f} mm")
print(f"Standard deviation: {weekly_precipitation.std():.2f} mm")
print(f"Max value is {z_score:.2f} standard deviations from the mean")


print(f"Highest daily water level: {daily_water_level.max()} cm (appeared {daily_water_level.idxmax()})")
save_figure(
    fig,
    "target_water_level_history",
    caption="Daily maximum water level at station 207241-at (in Korneuburg) over the full available period with weekly precipitation totals and the train/test split.",
)

## Figure — monthly data completeness by station

The heatmap shows the monthly percentage of available water level measurements for each station retained in the joined dataset. Interpolated measurements are counted as missing, so only directly observed measurements contribute to the displayed share. The bottom row shows the percentage of rows eligible for modeling; eligibility follows the joined feature contract.

In [ ]:
joined_columns = list(
    dict.fromkeys(
        [
            "timestamp",
            *water_level_columns,
            *(
                column.removesuffix("__water_level") + "__imputed"
                for column in water_level_columns
            ),
            dataset.contract.target_valid_column,
            *dataset.contract.predictor_columns,
            *dataset.contract.target_columns,
        ]
    )
)
joined_features = pd.concat(
    [
        pd.read_parquet(train_path, columns=joined_columns),
        pd.read_parquet(test_path, columns=joined_columns),
    ],
    ignore_index=True,
)
joined_features["timestamp"] = pd.to_datetime(
    joined_features["timestamp"], utc=True
)
joined_features["month"] = (
    joined_features["timestamp"].dt.tz_localize(None).dt.to_period("M")
)
station_ids = [
    column.removesuffix("__water_level") for column in water_level_columns
]
non_missing_percentages = {}
total_non_missing_percentages = {}
for station_id in station_ids:
    water_level_column = f"{station_id}__water_level"
    imputed_column = f"{station_id}__imputed"
    missing = (
        joined_features[water_level_column].isna()
        | joined_features[imputed_column].eq(True)
    )
    non_missing = ~missing
    non_missing_percentages[station_id] = (
        pd.DataFrame({"month": joined_features["month"], "non_missing": non_missing})
        .groupby("month")["non_missing"]
        .mean()
        .mul(100)
    )
    total_non_missing_percentages[station_id] = non_missing.mean() * 100

all_months = pd.period_range(
    joined_features["month"].min(),
    joined_features["month"].max(),
    freq="M",
)
missing_table = pd.DataFrame(
    {
        station_id: percentages.reindex(all_months)
        for station_id, percentages in non_missing_percentages.items()
    }
).T.reindex(station_ids)
eligible = (
    joined_features[dataset.contract.target_valid_column].eq(True)
    & joined_features[list(dataset.contract.predictor_columns)]
    .notna()
    .all(axis=1)
    & joined_features[list(dataset.contract.target_columns)].notna().all(axis=1)
)
total_eligible_percentage = eligible.mean() * 100
eligible_percentages = (
    pd.DataFrame({"month": joined_features["month"], "eligible": eligible})
    .groupby("month")["eligible"]
    .mean()
    .mul(100)
    .reindex(all_months)
)
eligible_row = pd.DataFrame(
    [eligible_percentages.to_numpy()],
    index=["Eligible rows"],
    columns=all_months,
)
completeness_table = pd.concat([missing_table, eligible_row])

cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("white")
fig = plt.figure(figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.58))
grid = fig.add_gridspec(
    2,
    3,
    height_ratios=(20, 1.2),
    width_ratios=(20, 4, 1),
    wspace=0.05,
    hspace=0.9,
)
axis = fig.add_subplot(grid[0, 0])
total_axis = fig.add_subplot(grid[0, 1], sharey=axis)
colorbar_axis = fig.add_subplot(grid[1, 0])
image = axis.imshow(
    completeness_table.to_numpy(dtype=float),
    aspect="auto",
    interpolation="none",
    cmap=cmap,
    vmin=0,
    vmax=100,
)
total_axis.set_xlim(0, 1)
total_axis.set_xticks([])
total_axis.tick_params(axis="y", left=False, labelleft=False)
total_axis.text(
    0.5,
    1.02,
    "Total",
    transform=total_axis.transAxes,
    ha="center",
    va="bottom",
    fontweight="bold",
)
for row, station_id in enumerate(station_ids):
    total_axis.text(
        0.5,
        row,
        f"{total_non_missing_percentages[station_id]:.1f}%",
        ha="center",
        va="center",
    )
total_axis.text(
    0.5,
    len(station_ids),
    f"{total_eligible_percentage:.1f}%",
    ha="center",
    va="center",
)
axis.set_title("Available water level\nmeasurements by month", loc="left")
axis.set_yticks(range(len(completeness_table)))
axis.set_yticklabels(completeness_table.index)
for label in axis.get_yticklabels():
    if label.get_text() == TARGET_STATION_ID:
        label.set_fontweight("bold")
axis.set_ylabel("Station / summary")
axis.axhline(len(station_ids) - 0.5, color="white", linewidth=1.5)
year_positions = [
    position
    for position, month in enumerate(all_months)
    if month.month == 1
]
axis.set_xticks(year_positions)
axis.set_xticklabels(
    [str(all_months[position].year) for position in year_positions],
    rotation=45,
    ha="right",
)
axis.set_xlabel("Month")
colorbar = fig.colorbar(
    image,
    cax=colorbar_axis,
    orientation="horizontal",
)
colorbar.set_label("Available water level measurements (%)", labelpad=2)
for spine in colorbar_axis.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(0.8)
# total_axis.axhline(len(station_ids) - 0.5, color="0.5", linewidth=1.0)
fig.subplots_adjust(left=0.27, right=0.98, top=0.86, bottom=0.25)
legend_bbox = colorbar_axis.get_position()
legend_frame = FancyBboxPatch(
    (legend_bbox.x0 - 0.025, legend_bbox.y0 - 0.12),
    legend_bbox.width + 0.05,
    legend_bbox.height + 0.12,
    boxstyle="round,pad=0.01,rounding_size=0.015",
    transform=fig.transFigure,
    facecolor="white",
    edgecolor=plt.rcParams["legend.edgecolor"],
    linewidth=0.8,
    alpha=plt.rcParams["legend.framealpha"],
    zorder=0,
)
fig.add_artist(legend_frame)
print(f"Nr of eligible rows: {eligible.sum()}")
save_figure(
    fig,
    "station_monthly_data_completeness",
    caption="Monthly available water level measurements and eligible rows for upstream stations (interpolated measurements count as missing).",
)

## Figure — cross-correlation between upstream stations and Korneuburg

For each water level column present in the joined feature Parquet files, this heatmap computes the pairwise-complete Pearson correlation with Korneuburg after shifting the upstream series by 0–72 hours. A positive lag therefore compares an upstream measurement with Korneuburg's level that many hours later; the marked maximum is a rough indication of propagation delay, not causal evidence.

In [ ]:
LAG_SCAN_HOURS = 72
feature_water_level_columns = [
    column
    for column in joined_features.columns
    if column.endswith("__water_level")
]
assert WATER_LEVEL_COLUMN in feature_water_level_columns
upstream_station_ids = [
    column.removesuffix("__water_level")
    for column in feature_water_level_columns
    if column != WATER_LEVEL_COLUMN
]
if not upstream_station_ids:
    raise ValueError("The joined feature artifacts contain no upstream water level columns")

correlation_input = joined_features.sort_values("timestamp").reset_index(drop=True)
target_series = correlation_input[WATER_LEVEL_COLUMN]
lag_hours = range(LAG_SCAN_HOURS + 1)
cross_correlation = pd.DataFrame(index=upstream_station_ids, columns=lag_hours, dtype=float)
for station_id in upstream_station_ids:
    upstream_series = correlation_input[f"{station_id}__water_level"]
    cross_correlation.loc[station_id] = [
        upstream_series.shift(lag).corr(target_series) for lag in lag_hours
    ]

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.48)
)
image = ax.imshow(
    cross_correlation.to_numpy(),
    aspect="auto",
    interpolation="none",
    # cmap="PuOr",
    vmin=0,
    vmax=1,
)
ax.set_yticks(range(len(upstream_station_ids)))
ax.set_yticklabels(upstream_station_ids)
lag_ticks = list(range(0, LAG_SCAN_HOURS + 1, 6))
ax.set_xticks(lag_ticks)
ax.set_xticklabels(lag_ticks)
ax.set_xlabel("Lag before Korneuburg (h)")
ax.set_ylabel("Upstream station")
ax.set_title("Upstream water level correlation with Korneuburg")
for row, station_id in enumerate(upstream_station_ids):
    best_lag = cross_correlation.loc[station_id].idxmax()
    if pd.notna(best_lag):
        print(f"Station {station_id} has best correlation at lag {best_lag} hours")
        ax.plot(
            best_lag,
            row,
            marker="o",
            markerfacecolor="none",
            markeredgecolor="black",
            markersize=5,
            markeredgewidth=0.8,
        )
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Pearson correlation")
fig.tight_layout()
save_figure(
    fig,
    "upstream_cross_correlation_heatmap",
    caption=f"Pearson cross-correlation between upstream stations and Korneuburg for lags of 0-{LAG_SCAN_HOURS} hours. Positive lag means the upstream measurement leads Korneuburg; circles mark each station's peak.",
)

## Figure — weather aggregates and future water level change

This RQ2 figure reproduces the descriptive Spearman correlations from the single-river EDA for the requested 6-, 24-, and 72-hour trailing precipitation sums and temperature means. Each cell relates the weather aggregate at issue time to the target station's water level change at the indicated future horizon; rows are paired only where both quantities are available.

In [ ]:
RELATIONSHIP_WINDOWS = (6, 24, 72)
RELATIONSHIP_HORIZONS = (1, 6, 12, 24, 48)
relationship_input = joined_features.sort_values("timestamp").reset_index(drop=True)
target_level = relationship_input[WATER_LEVEL_COLUMN]
future_changes = {
    horizon: target_level.shift(-horizon) - target_level
    for horizon in RELATIONSHIP_HORIZONS
}
relationship_records = []
for weather_label, column_prefix in [
    ("Precip.", "precipitation_rolling_sum"),
    ("Temp.", "temperature_2m_rolling_mean"),
]:
    for window in RELATIONSHIP_WINDOWS:
        aggregate_column = (
            f"{TARGET_STATION_ID}__{column_prefix}_{window}h"
        )
        row_label = f"{weather_label} last {window} h"
        for horizon, future_change in future_changes.items():
            pair = pd.concat(
                [
                    relationship_input[aggregate_column],
                    future_change.rename("future_change"),
                ],
                axis=1,
            ).dropna()
            relationship_records.append(
                {
                    "weather_aggregate": row_label,
                    "horizon_hours": horizon,
                    "spearman_rho": pair[aggregate_column].corr(
                        pair["future_change"], method="pearson"
                    ),
                }
            )
weather_relationships = pd.DataFrame(relationship_records)
relationship_matrix = weather_relationships.pivot(
    index="weather_aggregate",
    columns="horizon_hours",
    values="spearman_rho",
).reindex(
    index=[
        f"{weather} last {window} h"
        for weather in ("Precip.", "Temp.")
        for window in RELATIONSHIP_WINDOWS
    ],
    columns=RELATIONSHIP_HORIZONS,
)

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.48)
)
image = ax.imshow(
    relationship_matrix.to_numpy(),
    aspect="auto",
    interpolation="none",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
ax.set_xticks(range(len(RELATIONSHIP_HORIZONS)))
ax.set_xticklabels([f"+{horizon} h" for horizon in RELATIONSHIP_HORIZONS])
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
ax.set_xlabel("Future water level change", labelpad=8)
ax.set_yticks(range(len(relationship_matrix)))
ax.set_yticklabels(relationship_matrix.index)
ax.set_ylabel("Weather aggregate")
ax.set_title("Weather aggregates and future water level change", pad=34)
ax.axhline(2.5, color="white", linewidth=1.5)
for row in range(relationship_matrix.shape[0]):
    for column in range(relationship_matrix.shape[1]):
        value = relationship_matrix.iloc[row, column]
        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
        )
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Spearman correlation")
fig.tight_layout()
save_figure(
    fig,
    "weather_future_change_spearman",
    caption="Spearman correlations between precipitation sums or temperature means and future water level changes at the target station.",
)

## Figure — observed water level distribution

A violin plot with an overlaid boxplot of every observed water level at the target station, authored at exactly the thesis text width. `save_figure` writes the PDF and prints the `figure` float; the figure stays open so it also renders inline here.

In [ ]:
observed_water_level = water_level.dropna()
quartiles = observed_water_level.quantile([0.25, 0.50, 0.75])
above_alarm_percentage = (
    observed_water_level.gt(WATER_LEVEL_ALARM_THRESHOLD_CM).mean() * 100
)
above_alarm_count = observed_water_level.gt(WATER_LEVEL_ALARM_THRESHOLD_CM).sum()
fig, ax = plt.subplots(figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55))
violin = ax.violinplot(
    observed_water_level,
    positions=[1],
    orientation="horizontal",
    showextrema=False,
)
for body in violin["bodies"]:
    body.set_alpha(0.75)
ax.boxplot(
    observed_water_level,
    positions=[1],
    widths=0.15,
    orientation="horizontal",
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markerfacecolor": "none", "markeredgecolor": "black", "markersize": 3, "linestyle": "none", "alpha": 0.6, "zorder": 5},
)

ax.axvline(
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    color="C2",
    linestyle="--",
    label=f"Alarm threshold ({WATER_LEVEL_ALARM_THRESHOLD_CM:.0f} cm)",
)
ax.set_xlabel("Water level [cm]")
ax.set_title(f"Observed water level at {TARGET_STATION_ID}")
ax.set_yticks([])
ax.tick_params(axis="y", left=False, labelleft=False)
ax.grid(alpha=0.25, axis="x")
# ax.text(
#     0.98,
#     0.95,
#     f"{above_alarm_percentage:.1f}% of observations\nabove alarm threshold",
#     transform=ax.transAxes,
#     ha="right",
#     va="top",
#     bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "0.5", "alpha": 0.9},
# )
print(f"Number of observations above alarm threshold: {above_alarm_count} ({above_alarm_percentage:.2f}%)")
print(f"Quartiles: Q1={quartiles[0.25]:.2f}, Q2={quartiles[0.50]:.2f}, Q3={quartiles[0.75]:.2f}")
ax.legend()
save_figure(
    fig,
    "target_water_level_distribution",
    caption="Distribution of observed hourly water levels at the target station.",
)

## Table — processed data coverage by station

One row per station in the joined train and test data. `start date` and `end date` are the first and last timestamps with a non-null water-level value; `missing count` is the number of null water level rows; `largest gap (hours)` is the largest elapsed gap between consecutive non-missing water-level observations.

In [ ]:
joined_data = pd.concat(
    [
        pd.read_parquet(raw_train_path, columns=["timestamp", *water_level_columns]),
        pd.read_parquet(raw_test_path, columns=["timestamp", *water_level_columns]),
    ],
    ignore_index=True,
).sort_values("timestamp")
timestamps = pd.to_datetime(joined_data["timestamp"], utc=True)
coverage_rows = []
for column in water_level_columns:
    station_id = column.removesuffix("__water_level")
    water_level = joined_data[column]
    observed_timestamps = timestamps[water_level.notna()]
    largest_gap_hours = (
        observed_timestamps.diff().dt.total_seconds().div(3600).max()
    )
    coverage_rows.append(
        {
            "Station": station_id,
            "Start date": observed_timestamps.min().strftime("%Y-%m-%d %H:%M"),
            "End date": observed_timestamps.max().strftime("%Y-%m-%d %H:%M"),
            # "Row count": len(joined_data),
            "Missing count": int(water_level.isna().sum()),
            "Largest gap (h)": int(largest_gap_hours),
        }
    )

station_coverage = pd.DataFrame(coverage_rows)
display(station_coverage)
# save_table(
#     station_coverage,
#     "station_data_coverage",
#     caption=f"Coverage of water level data across stations. (Total row count: {len(joined_data):,})",
#     index=False
# )

## Table — hyperparameter tuning results

Loads the newest complete model executions from MLflow and reports each selected candidate's feature subset, hyperparameters, and aggregate cross-validation RMSE. Models without a complete MLflow execution remain as placeholders.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

from src.config import MLFLOW_TRACKING_URI
from src.evaluation import (
    FEATURE_SUBSET_MODEL_EXPERIMENTS,
    MODEL_EXPERIMENTS,
    load_latest_complete_comparison_metrics,
    load_latest_complete_feature_subset_cv_metrics,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_client = MlflowClient(MLFLOW_TRACKING_URI)
model_experiments = {
    model: experiment
    for model, experiment in MODEL_EXPERIMENTS.items()
    if model != "Random Forest"
}
comparison_metrics = load_latest_complete_comparison_metrics(
    client=mlflow_client,
    model_experiments=model_experiments,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
selected_cv = comparison_metrics.loc[
    comparison_metrics["phase"].eq("cross_validation")
    & comparison_metrics["scope"].eq("aggregate")
    & comparison_metrics["metric"].eq("rmse")
].copy()
if selected_cv["model"].duplicated().any():
    raise ValueError("MLflow returned multiple selected CV rows for a model")

FEATURE_SUBSET_LABELS = {
    "full": "Full",
    "all_station_hydrology_quality_time": "All-station hydrology, quality, and time",
    "raw_all_stations": "Raw all stations",
    "target_station_full": "Target station full",
    "target_station_hydrology_quality_time": "Target-station hydrology, quality, and time",
    "current_water_levels_all_stations": "Current water levels from all stations",
}


def format_mlflow_float(value: str) -> str:
    return f"{float(value):.6g}"


def required_param(params: dict[str, str], name: str) -> str:
    value = params.get(name)
    if value is None:
        raise ValueError(f"Selected MLflow run is missing parameter {name!r}")
    return value


def selected_row(model: str, cv_row: pd.Series) -> dict[str, str]:
    params = mlflow_client.get_run(str(cv_row["run_id"])).data.params
    if model == "RNN":
        subset = "Raw all stations"
    elif model == "Persistence":
        subset = "Not applicable"
    else:
        subset_name = required_param(params, "subset")
        subset = FEATURE_SUBSET_LABELS.get(subset_name, subset_name)
    if model == "Persistence":
        hyperparameters = "Not applicable"
    elif model == "Ridge":
        hyperparameters = (
            f"alpha={format_mlflow_float(required_param(params, 'alpha'))}, "
            f"log1p={required_param(params, 'log1p')}"
        )
    elif model == "MLP":
        hyperparameters = (
            f"layers={required_param(params, 'hidden_layer_sizes')}, "
            f"alpha={format_mlflow_float(required_param(params, 'alpha'))}"
        )
    elif model == "XGBoost":
        hyperparameters = (
            f"depth={required_param(params, 'max_depth')}, "
            f"learning_rate={format_mlflow_float(required_param(params, 'learning_rate'))}"
        )
    elif model in {"Random Forest", "Extra Trees"}:
        hyperparameters = (
            f"depth={required_param(params, 'max_depth')}, "
            f"n_estimators={required_param(params, 'n_estimators')}, "
            f"min_leaf={required_param(params, 'min_samples_leaf')}, "
            f"max_features={required_param(params, 'max_features')}"
        )
    elif model == "RNN":
        hyperparameters = (
            f"cell={required_param(params, 'cell_type')}, "
            f"sequence_length={required_param(params, 'sequence_length')}"
        )
    else:
        raise ValueError(f"Unsupported model {model!r}")
    return {
        "Model": model,
        "Selected feature subset": subset,
        "Selected hyperparameters": hyperparameters,
        "CV RMSE": f"{float(cv_row['value']):.2f}",
    }


hyperparameter_tuning_rows = []
for model in model_experiments:
    model_rows = selected_cv.loc[selected_cv["model"].eq(model)]
    if model_rows.empty:
        print(f"No complete MLflow execution found for {model}; leaving placeholders.")
        hyperparameter_tuning_rows.append(
            {
                "Model": model,
                "Selected feature subset": (
                    "Raw all stations"
                    if model == "RNN"
                    else "Not applicable"
                    if model == "Persistence"
                    else "..."
                ),
                "Selected hyperparameters": "...",
                "CV RMSE": "...",
            }
        )
    else:
        hyperparameter_tuning_rows.append(selected_row(model, model_rows.iloc[0]))

hyperparameter_tuning_results = pd.DataFrame(hyperparameter_tuning_rows)
hyperparameter_tuning_results.sort_values("CV RMSE", inplace=True)
display(hyperparameter_tuning_results)
hyperparameter_tuning_table_path = save_table(
    hyperparameter_tuning_results,
    "hyperparameter_tuning_results",
    caption=f"Selected feature subsets, hyperparameters, and \\ac{{CV}} \\ac{{RMSE}} for the tuned forecasting models.",
    index=False,
    environment="tabularx",
    column_format="@{}lXXr@{}",
    addlinespace=True,
)

## Figure — aggregate cross-validation RMSE comparison

This Matplotlib figure reproduces the aggregate cross-validation comparison from `05_evaluate.ipynb` for the RMSE metric only. Each marker is the selected candidate's mean across validation folds; error bars show one logged fold-to-fold standard deviation.

In [ ]:
rmse_rows = selected_cv.reset_index(drop=True)
if rmse_rows.empty:
    raise ValueError("No aggregate cross-validation RMSE rows were found")
rmse_rows.sort_values("value", inplace=True)
models = rmse_rows["model"].astype(str).tolist()
values = rmse_rows["value"].to_numpy(dtype=float)
standard_deviations = rmse_rows["cv_std"].to_numpy(dtype=float)
positions = range(len(models))

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55)
)
for position, model, value, standard_deviation in zip(
    positions, models, values, standard_deviations, strict=True
):
    print(f"Model {model}: RMSE={value:.2f} ± {standard_deviation:.2f}")
    ax.errorbar(
        position,
        value,
        yerr=standard_deviation,
        fmt="o",
        color=f"C{position % 10}",
        capsize=2,
        # markersize=5,
        # linewidth=1,
    )

ax.set_xticks(list(positions), labels=models, rotation=-20, ha="left")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE (cm)")
ax.set_title("Aggregate Cross-Validation RMSE (mean $\\pm$ standard deviation)")
ax.grid(alpha=0.25, axis="y")
fig.tight_layout()
cv_rmse_figure_path = save_figure(
    fig,
    "aggregate_cross_validation_rmse",
    caption=f"Aggregate \\ac{{CV}} \\ac{{RMSE}} of all trained models. The marker shows the mean across 5 folds, the error bar shows the standard deviation within the folds.",
)

## Figure — effect of input features on cross-validation RMSE

For each subset-aware model, this Matplotlib figure shows the best candidate within each feature subset. Markers show aggregate cross-validation RMSE means, error bars show fold-to-fold standard deviations, and the dashed line is the persistence baseline repeated across subsets.

In [ ]:
FEATURE_SUBSET_MODELS = ("XGBoost",)
# For example: FEATURE_SUBSET_MODELS = ("XGBoost", "MLP")
if not FEATURE_SUBSET_MODELS:
    raise ValueError("Select at least one feature-subset model")
unsupported_models = sorted(
    set(FEATURE_SUBSET_MODELS).difference(FEATURE_SUBSET_MODEL_EXPERIMENTS)
)
if unsupported_models:
    raise ValueError(f"Unknown feature-subset models: {unsupported_models}")
unavailable_models = sorted(set(FEATURE_SUBSET_MODELS).difference(model_experiments))
if unavailable_models:
    raise ValueError(
        f"Feature-subset models are not in the selected MLflow scope: {unavailable_models}"
    )
feature_subset_model_experiments = {
    model: experiment
    for model, experiment in FEATURE_SUBSET_MODEL_EXPERIMENTS.items()
    if model in model_experiments and model in FEATURE_SUBSET_MODELS
}
feature_subset_metrics = load_latest_complete_feature_subset_cv_metrics(
    client=mlflow_client,
    model_experiments=feature_subset_model_experiments,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
best_feature_subset_rows = feature_subset_metrics.loc[
    feature_subset_metrics["is_best_within_subset"].eq(True)
].copy()
if best_feature_subset_rows.empty:
    raise ValueError("No best feature-subset candidate rows were found")
if best_feature_subset_rows.duplicated(["model", "subset"]).any():
    raise ValueError("Best feature-subset rows duplicate a model/subset")

feature_subset_order = [
    "full",
    "all_station_hydrology_quality_time",
    "raw_all_stations",
    "target_station_full",
    "target_station_hydrology_quality_time",
    "current_water_levels_all_stations",
]
observed_subsets = (
    best_feature_subset_rows["subset"].astype(str).drop_duplicates().tolist()
)
subsets = [
    subset for subset in feature_subset_order if subset in observed_subsets
] + [
    subset for subset in observed_subsets if subset not in feature_subset_order
]
subset_positions = {subset: position for position, subset in enumerate(subsets)}
subset_labels = {
    "full": "Full",
    "all_station_hydrology_quality_time": "All-station\nhydrology +\nquality/time",
    "raw_all_stations": "Raw all\nstations",
    "target_station_full": "Target\nstation\nfull",
    "target_station_hydrology_quality_time": "Target\nhydrology +\nquality/time",
    "current_water_levels_all_stations": "Current water\nlevels,\nall stations",
}

persistence_rows = selected_cv.loc[selected_cv["model"].eq("Persistence")]
if len(persistence_rows) != 1:
    raise ValueError("Expected exactly one persistence RMSE baseline row")
persistence_value = float(persistence_rows.iloc[0]["value"])
persistence_standard_deviation = float(persistence_rows.iloc[0]["cv_std"])

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.72)
)
baseline_positions = list(range(len(subsets)))
ax.errorbar(
    baseline_positions,
    [persistence_value] * len(subsets),
    yerr=[persistence_standard_deviation] * len(subsets),
    fmt="none",
    ecolor="0.45",
    elinewidth=1,
    capsize=2,
    zorder=1,
)
ax.plot(
    baseline_positions,
    [persistence_value] * len(subsets),
    linestyle="--",
    color="0.45",
    linewidth=1.2,
    label="Persistence baseline",
    zorder=1,
)

model_markers = ("o", "s", "^", "D", "P")
models = best_feature_subset_rows["model"].astype(str).drop_duplicates().tolist()
for position, model in enumerate(models):
    marker = model_markers[position % len(model_markers)]
    model_rows = best_feature_subset_rows.loc[
        best_feature_subset_rows["model"].eq(model)
    ].copy()
    model_rows["subset_order"] = model_rows["subset"].map(subset_positions)
    model_rows.sort_values("subset_order", inplace=True)
    x_values = model_rows["subset_order"].to_numpy(dtype=float)
    ax.errorbar(
        x_values,
        model_rows["cv_rmse_mean"].to_numpy(dtype=float),
        yerr=model_rows["cv_rmse_std"].to_numpy(dtype=float),
        fmt=f"{marker}-",
        color=f"C{position}",
        capsize=2,
        linewidth=1.1,
        markersize=4.5,
        label=model,
        zorder=2,
    )

ax.set_xticks(
    list(range(len(subsets))),
    labels=[
        subset_labels.get(subset, subset.replace("_", " ").title())
        for subset in subsets
    ],
)
ax.set_xlabel("Input feature subset")
ax.set_ylabel("CV RMSE (cm)")
ax.set_title("Cross-validation RMSE by feature subset and model")
ax.grid(alpha=0.25, axis="y")
fig.subplots_adjust(left=0.12, right=0.98, top=0.88, bottom=0.32)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=3,
)
feature_subset_rmse_figure_path = save_figure(
    fig,
    "feature_subset_cross_validation_rmse",
    caption=f"\\ac{{CV}} \\ac{{RMSE}} by input feature subset for the best candidate within each subset. Markers show the mean across {N_VALIDATION_FOLDS} folds, error bars show the fold-to-fold standard deviation, and the dashed line shows the persistence baseline.",
)

## Table — controlled feature-subset performance comparison

The table reports the selected subset-aware model for each controlled input configuration. Set `FEATURE_SUBSET_COMPARISON_MODEL` in the code cell below to choose the model. The rows use the pipeline's current subset definitions: `target_station_hydrology_quality_time` (target station only), `all_station_hydrology_quality_time` (add neighboring-station hydrology, quality, and time features), `target_station_full` (add weather to the target station), and `full` (all-station features, including weather). Changes are relative to the target-station-only row.

The target-station-only versus target-station-full comparison isolates weather, as does all-station hydrology/quality/time versus `full` for the all-station contract. The target-station-full versus `full` comparison adds neighboring-station features and neighboring weather together, so it is not interpreted as an isolated neighboring-station effect.

In [ ]:
FEATURE_SUBSET_COMPARISON_MODEL = "XGBoost"
if FEATURE_SUBSET_COMPARISON_MODEL not in FEATURE_SUBSET_MODEL_EXPERIMENTS:
    raise ValueError(
        f"Unknown subset-aware model: {FEATURE_SUBSET_COMPARISON_MODEL!r}"
    )
comparison_subset_order = {
    "Target station only": "target_station_hydrology_quality_time",
    "+ neighboring stations": "all_station_hydrology_quality_time",
    "+ weather": "target_station_full",
    "+ neighboring stations + weather": "full",
}
table_model_experiments = {
    FEATURE_SUBSET_COMPARISON_MODEL: FEATURE_SUBSET_MODEL_EXPERIMENTS[
        FEATURE_SUBSET_COMPARISON_MODEL
    ]
}
table_feature_subset_metrics = load_latest_complete_feature_subset_cv_metrics(
    client=mlflow_client,
    model_experiments=table_model_experiments,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
table_best_candidates = table_feature_subset_metrics.loc[
    table_feature_subset_metrics["is_best_within_subset"].eq(True)
].copy()
if table_best_candidates.empty:
    raise ValueError("No best feature-subset candidates were found for the table")
if table_best_candidates["model"].ne(FEATURE_SUBSET_COMPARISON_MODEL).any():
    raise ValueError("The feature-subset metrics contain an unexpected model")
if table_best_candidates.duplicated(["model", "subset"]).any():
    raise ValueError("Best feature-subset candidates duplicate a model/subset")

comparison_rows = []
for comparison, subset in comparison_subset_order.items():
    candidates = table_best_candidates.loc[
        table_best_candidates["subset"].eq(subset)
    ].sort_values("cv_rmse_mean", kind="stable")
    if candidates.empty:
        raise ValueError(
            f"No complete model execution contains subset {subset!r}"
        )
    best_candidate = candidates.iloc[0]
    comparison_rows.append(
        {
            "Comparison": comparison,
            "Model": FEATURE_SUBSET_COMPARISON_MODEL,
            "CV RMSE": float(best_candidate["cv_rmse_mean"]),
        }
    )

comparison_table = pd.DataFrame(comparison_rows)
reference_rmse = float(comparison_table.iloc[0]["CV RMSE"])
if not reference_rmse > 0:
    raise ValueError("Reference CV RMSE must be positive")

def format_rmse_change(value: float) -> str:
    change = (value / reference_rmse - 1) * 100
    sign = "-" if change < 0 else "+" if change > 0 else ""
    return f"{sign}{abs(change):.1f}%"

comparison_table["Change in RMSE"] = [
    "reference",
    *(format_rmse_change(value) for value in comparison_table["CV RMSE"].iloc[1:]),
]
display(comparison_table)
feature_subset_comparison_table_path = save_table(
    comparison_table,
    "feature_subset_performance_comparison",
    caption=f"\\ac{{CV}} \\ac{{RMSE}} for {FEATURE_SUBSET_COMPARISON_MODEL} under controlled feature-subset configurations.",
    index=False,
    column_format="@{}Xlrr@{}",
    addlinespace=True,
)

## Figure — sealed-test performance by forecast horizon

This prominent figure compares sealed-test RMSE for the selected model and the persistence baseline across every forecast horizon. Set `SELECTED_MODEL` at the top of the code cell to choose the winning model.

In [ ]:
SELECTED_MODEL = "XGBoost"

import numpy as np

available_models = set(selected_cv["model"].astype(str))
available_forecasting_models = sorted(available_models.difference({"Persistence"}))
if SELECTED_MODEL == "Persistence":
    raise ValueError("SELECTED_MODEL must be a forecasting model, not Persistence")
if SELECTED_MODEL not in available_models:
    raise ValueError(
        f"SELECTED_MODEL {SELECTED_MODEL!r} is not available; choose from {available_forecasting_models}"
    )
selected_model = SELECTED_MODEL
models = (selected_model, "Persistence")
metrics = ("rmse",)
horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
horizon_rows = comparison_metrics.loc[
    comparison_metrics["phase"].eq("sealed_test")
    & comparison_metrics["scope"].eq("horizon")
    & comparison_metrics["model"].isin(models)
    & comparison_metrics["metric"].isin(metrics)
].copy()
if horizon_rows.duplicated(["model", "metric", "horizon"]).any():
    raise ValueError("Sealed-test horizon rows duplicate a model/metric/horizon")

horizon_values = {}
for model in models:
    for metric in metrics:
        metric_rows = horizon_rows.loc[
            horizon_rows["model"].eq(model)
            & horizon_rows["metric"].eq(metric)
        ].sort_values("horizon", kind="stable")
        if metric_rows["horizon"].astype(int).tolist() != horizons:
            raise ValueError(
                f"Sealed-test {metric.upper()} rows do not cover every horizon for {model}"
            )
        values = metric_rows["value"].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"Sealed-test {metric.upper()} values are not finite for {model}")
        horizon_values[(model, metric)] = values

fig, ax = plt.subplots(
    figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55)
)
ax.plot(
    horizons,
    horizon_values[(selected_model, metric)],
    marker="o",
    markersize=3.5,
    linewidth=1.4,
    label=selected_model,
    zorder=2,
)
ax.plot(
    horizons,
    horizon_values[("Persistence", metric)],
    marker="o",
    markersize=2.5,
    linestyle="--",
    color="0.45",
    linewidth=1.2,
    label="Persistence baseline",
    zorder=1,
)
ax.set_title(f"Test set RMSE by forecast horizon ({selected_model})")
ax.set_xlabel("Forecast horizon (hours)")
ax.set_ylabel("RMSE (cm)")
ax.set_xlim(1, FORECAST_HORIZON_HOURS)
ax.set_xticks((1, 4, 8, 12, 16, 20, 24))
ax.grid(alpha=0.25)

fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.22)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
)
sealed_test_horizon_figure_path = save_figure(
    fig,
    "sealed_test_performance_by_horizon",
    caption=f"Test set \\ac{{RMSE}} across the 24-hour forecast horizon for the selected {selected_model} model and the persistence baseline.",
)

## Using the exports in the thesis

Each `save_*` call prints the float to paste into a chapter, with the path already relative to the thesis repository root — copy it as-is. The figure is authored at exactly `\textwidth`, so `width=\textwidth` scales it by 1.0 and its fonts match the body text.